# arXiv Dataset — Ryan's EDA

**Branch:** `ryan`  
**Dataset:** arXiv Metadata OAI Snapshot  

## Unique Analysis Angles

This notebook explores four dimensions of the arXiv dataset that are *not* covered by other team members:

| Section | Analysis | What It Answers |
|---------|----------|-----------------|
| 1 | **Category Co-occurrence Network** | Which research fields overlap most? What are the cross-disciplinary hubs? |
| 2 | **LDA Topic Modeling** | What latent topics exist across all abstracts (probabilistic, not cluster-based)? |
| 3 | **Submission Calendar Patterns** | When do researchers publish? Day-of-week, monthly, and yearly rhythms. |
| 4 | **Author Productivity Analysis** | Who are the most prolific authors? Do top authors span multiple fields? |

> **Note:** King's branch covers word-frequency EDA + TF-IDF + K-Means clustering.  
> **Note:** Geon's branch covers LLM keyword time-series tracking + ARIMA/Prophet forecasting.  
> This notebook deliberately avoids all of those approaches.

## 0. Setup & Data Loading

In [1]:
import json
import re
import random
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as mticker

# NLP
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Network
import networkx as nx

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
random.seed(42)
np.random.seed(42)
print('Libraries loaded.')

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Load full dataset — same approach as arxiv_project.ipynb
import os

df = pd.read_json('./arxiv-metadata-oai-snapshot.json', lines=True)
print(f'Loaded {len(df):,} rows')
df.head(3)

In [ ]:
# Quick schema check
print(df.dtypes)
print(f'\nShape: {df.shape}')
print(f'Null counts:\n{df.isnull().sum()}')

---
## Section 1 — Category Co-occurrence Network

**Question:** Which research fields are most commonly combined in a single paper?  
**Method:** Build a weighted undirected graph where nodes = arXiv categories and edge weights = number of papers spanning both categories. Analyse degree centrality, community structure, and the top cross-disciplinary bridges.

This is fundamentally different from clustering *abstracts* (King's approach) — here we cluster *fields* based on how often researchers bridge them.

In [ ]:
# Parse multi-category papers
# categories column is a space-separated string, e.g. 'cs.LG stat.ML'
df['cat_list'] = df['categories'].fillna('').str.split()
df['primary_cat'] = df['cat_list'].apply(lambda x: x[0] if x else None)
multi_cat = df[df['cat_list'].apply(len) > 1]
print(f'Papers with ≥2 categories: {len(multi_cat):,} ({len(multi_cat)/len(df)*100:.1f}%)')

In [ ]:
# Build co-occurrence edge list
edge_counter = Counter()
for cat_list in multi_cat['cat_list']:
    for pair in combinations(sorted(set(cat_list)), 2):
        edge_counter[pair] += 1

# Keep only pairs that co-occur >= 50 times to keep the graph readable
MIN_WEIGHT = 50
edges = [(a, b, w) for (a, b), w in edge_counter.items() if w >= MIN_WEIGHT]
print(f'Edges with weight >= {MIN_WEIGHT}: {len(edges):,}')

In [ ]:
# Build NetworkX graph
G = nx.Graph()
for a, b, w in edges:
    G.add_edge(a, b, weight=w)

print(f'Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}')

# Degree centrality — which category appears in the most cross-field papers
deg_centrality = nx.degree_centrality(G)
top_nodes = sorted(deg_centrality, key=deg_centrality.get, reverse=True)[:15]
print('\nTop 15 categories by degree centrality:')
for n in top_nodes:
    print(f'  {n:20s}  {deg_centrality[n]:.4f}')

In [ ]:
# ---- Plot 1a: Degree centrality bar chart ----
top20 = sorted(deg_centrality, key=deg_centrality.get, reverse=True)[:20]
vals   = [deg_centrality[n] for n in top20]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top20[::-1], vals[::-1], color='steelblue')
ax.set_xlabel('Degree Centrality')
ax.set_title('Top 20 arXiv Categories by Cross-Field Degree Centrality', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 1b: Network graph (top 40 nodes by degree) ----
top40 = sorted(deg_centrality, key=deg_centrality.get, reverse=True)[:40]
H = G.subgraph(top40)

# Layout
pos = nx.spring_layout(H, seed=42, k=1.8)

# Node sizes proportional to weighted degree
wdeg = dict(H.degree(weight='weight'))
node_sizes = [wdeg[n] / 30 for n in H.nodes()]

# Edge widths proportional to weight
weights = [H[u][v]['weight'] for u, v in H.edges()]
max_w   = max(weights)
edge_widths = [2 + 6 * (w / max_w) for w in weights]

# Colour by top-level domain prefix
domain_colors = {
    'cs': '#1f77b4', 'math': '#ff7f0e', 'physics': '#2ca02c',
    'stat': '#d62728', 'astro': '#9467bd', 'cond': '#8c564b',
    'quant': '#e377c2', 'hep': '#17becf', 'econ': '#bcbd22',
    'q-bio': '#7f7f7f',
}
def node_color(n):
    prefix = n.split('.')[0].split('-')[0]
    return domain_colors.get(prefix, '#aaaaaa')

node_colors = [node_color(n) for n in H.nodes()]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_nodes(H, pos, node_size=node_sizes, node_color=node_colors, alpha=0.85, ax=ax)
nx.draw_networkx_edges(H, pos, width=edge_widths, alpha=0.25, edge_color='#555555', ax=ax)
nx.draw_networkx_labels(H, pos, font_size=7, ax=ax)

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(color=c, label=d) for d, c in domain_colors.items()]
ax.legend(handles=legend_els, loc='lower left', fontsize=8, framealpha=0.7)
ax.set_title('Category Co-occurrence Network (top 40 nodes, edge weight ≥ 50)', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 1c: Top 20 strongest cross-field bridges ----
top_edges = sorted(edges, key=lambda x: x[2], reverse=True)[:20]

labels = [f'{a}  ↔  {b}' for a, b, _ in top_edges]
weights_plot = [w for _, _, w in top_edges]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels[::-1], weights_plot[::-1], color='coral')
ax.set_xlabel('Number of Co-occurring Papers')
ax.set_title('Top 20 Strongest Category Pair Co-occurrences', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Takeaways ----
print('=== Section 1 Takeaways ===')
print(f'Most cross-disciplinary category (by degree centrality): {top_nodes[0]}')
print(f'Strongest category bridge: {top_edges[0][0]} <-> {top_edges[0][1]}  ({top_edges[0][2]:,} papers)')
print(f'Total cross-field pairs with ≥50 papers: {len(edges):,}')

---
## Section 2 — LDA Topic Modeling

**Question:** What latent topics exist across arXiv abstracts — and how are papers distributed across them?  
**Method:** Latent Dirichlet Allocation (LDA) on a bag-of-words representation of cleaned abstracts.

**How this differs from King's K-Means:** K-Means assigns each paper to exactly one cluster based on its TF-IDF vector distance. LDA models each paper as a *mixture* of topics — a paper can be 60% physics + 40% ML. This gives a probabilistic, interpretable decomposition rather than hard cluster assignments.

**How this differs from Geon's work:** Geon tracks *specific pre-defined keywords* over time. LDA discovers topics *from the data itself*, with no prior assumptions about what matters.

In [ ]:
# Subsample for LDA (keep it tractable)
LDA_SAMPLE = min(100_000, len(df))
lda_df = df.dropna(subset=['abstract']).sample(LDA_SAMPLE, random_state=42).copy()
print(f'LDA sample size: {len(lda_df):,}')

# Lightweight preprocessing — remove LaTeX, lowercase, strip numbers
def clean_abstract(text):
    text = re.sub(r'\$.*?\$', ' ', text)        # inline LaTeX
    text = re.sub(r'\\[a-z]+\{.*?\}', ' ', text) # LaTeX commands
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    return re.sub(r'\s+', ' ', text).strip()

lda_df['abstract_clean'] = lda_df['abstract'].apply(clean_abstract)
print('Cleaning done.')

In [ ]:
# Custom stopwords — generic academic/arXiv terms that don't convey topic identity
CUSTOM_STOP = [
    'paper', 'propose', 'proposed', 'show', 'shows', 'shown', 'present',
    'result', 'results', 'method', 'methods', 'approach', 'approaches',
    'based', 'using', 'used', 'use', 'study', 'work', 'new', 'model',
    'models', 'problem', 'data', 'analysis', 'two', 'also', 'first',
    'performance', 'effective', 'provide', 'significant', 'set', 'well',
    'large', 'high', 'however', 'number', 'different', 'general',
    'consider', 'known', 'can', 'may', 'one', 'order', 'system',
    'function', 'framework', 'task', 'learning', 'training'
]

vectorizer = CountVectorizer(
    max_df=0.85,
    min_df=50,
    max_features=5000,
    stop_words='english',
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b',
)

X = vectorizer.fit_transform(lda_df['abstract_clean'])

# Remove custom stopwords post-fit by zeroing their columns
vocab = np.array(vectorizer.get_feature_names_out())
stop_idx = [i for i, w in enumerate(vocab) if w in CUSTOM_STOP]
X[:, stop_idx] = 0

print(f'Vocabulary size: {len(vocab):,} | Matrix: {X.shape}')

In [ ]:
# Fit LDA — 15 topics
N_TOPICS = 15

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=15,
    learning_method='online',
    batch_size=2048,
    random_state=42,
    n_jobs=-1,
)
lda.fit(X)
print(f'Log-likelihood: {lda.score(X):.1f}')
print(f'Perplexity:     {lda.perplexity(X):.1f}')

In [ ]:
# ---- Plot 2a: Top words per topic ----
N_TOP_WORDS = 12

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
colors = cm.tab20(np.linspace(0, 1, N_TOPICS))

for topic_idx, (component, ax) in enumerate(zip(lda.components_, axes)):
    top_words_idx = component.argsort()[-N_TOP_WORDS:]
    top_words     = vocab[top_words_idx]
    word_weights  = component[top_words_idx]
    ax.barh(top_words, word_weights, color=colors[topic_idx])
    ax.set_title(f'Topic {topic_idx + 1}', fontweight='bold', fontsize=10)
    ax.tick_params(labelsize=8)
    ax.set_xlabel('Weight', fontsize=8)

fig.suptitle(f'LDA — Top {N_TOP_WORDS} Words per Topic ({N_TOPICS} topics)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Assign dominant topic to each paper
doc_topic = lda.transform(X)
lda_df = lda_df.copy()
lda_df['dominant_topic'] = doc_topic.argmax(axis=1)
lda_df['topic_weight']   = doc_topic.max(axis=1)

topic_dist = lda_df['dominant_topic'].value_counts().sort_index()
print('Papers per dominant topic:')
print(topic_dist.to_string())

In [ ]:
# ---- Plot 2b: Paper distribution across topics ----
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(topic_dist.index + 1, topic_dist.values, color=colors[:N_TOPICS])
ax.set_xlabel('Topic Number')
ax.set_ylabel('Number of Papers (dominant topic)')
ax.set_title('Distribution of Papers Across LDA Topics', fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 2c: Topic mixture heatmap for a random 500-paper slice ----
slice_idx = np.random.choice(len(doc_topic), 500, replace=False)
heatmap_data = doc_topic[slice_idx]
# Sort rows by dominant topic for cleaner visual
heatmap_data = heatmap_data[heatmap_data.argmax(axis=1).argsort()]

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(heatmap_data.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Papers (sorted by dominant topic)')
ax.set_ylabel('Topic')
ax.set_yticks(range(N_TOPICS))
ax.set_yticklabels([f'T{i+1}' for i in range(N_TOPICS)], fontsize=8)
plt.colorbar(im, ax=ax, label='Topic Probability')
ax.set_title('Topic Mixture Heatmap — 500 Random Papers\n(rows = topics, cols = papers sorted by dominant topic)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 2d: Topic confidence distribution ----
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lda_df['topic_weight'], bins=50, edgecolor='white', color='teal')
ax.axvline(lda_df['topic_weight'].mean(), color='red', linestyle='--', label=f'Mean = {lda_df["topic_weight"].mean():.2f}')
ax.set_xlabel('Dominant Topic Probability')
ax.set_ylabel('Number of Papers')
ax.set_title('How Confidently Does Each Paper Belong to Its Dominant Topic?', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 3 — Submission Calendar Patterns

**Question:** When do researchers submit to arXiv? Are there weekly rhythms, monthly surges, or long-term growth trends that reveal something about academic culture?

**How this differs from Geon's work:** Geon tracked *what* researchers write about over time (LLM keyword frequencies). This section asks *when* they submit — temporal rhythms independent of content.

In [ ]:
# Parse update_date
# update_date format: YYYY-MM-DD
cal_df = df.dropna(subset=['update_date']).copy()
cal_df['date'] = pd.to_datetime(cal_df['update_date'], errors='coerce')
cal_df = cal_df.dropna(subset=['date'])

cal_df['year']       = cal_df['date'].dt.year
cal_df['month']      = cal_df['date'].dt.month
cal_df['dayofweek']  = cal_df['date'].dt.dayofweek   # 0=Mon … 6=Sun
cal_df['week']       = cal_df['date'].dt.isocalendar().week.astype(int)

print(f'Date range: {cal_df["date"].min().date()}  →  {cal_df["date"].max().date()}')
print(f'Total papers with valid dates: {len(cal_df):,}')

In [ ]:
# ---- Plot 3a: Annual submission volume ----
yearly = cal_df.groupby('year').size()
yearly = yearly[(yearly.index >= 1991) & (yearly.index <= 2024)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(yearly.index, yearly.values, color='steelblue', width=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Papers')
ax.set_title('Annual arXiv Submission Volume (all fields)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3b: Monthly seasonality ----
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = cal_df.groupby('month').size()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(1, 13), monthly.values, color='coral', tick_label=month_names)
ax.set_ylabel('Total Papers')
ax.set_title('Monthly Submission Volume (all years, all fields)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3c: Day-of-week rhythm ----
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow = cal_df.groupby('dayofweek').size()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(7), dow.values, color=['#2196F3']*5 + ['#FF5722']*2, tick_label=dow_names)
ax.set_ylabel('Total Papers')
ax.set_title('Day-of-Week Submission Pattern', fontweight='bold')
ax.annotate('Weekend\ndrop-off', xy=(5, dow.iloc[5]), xytext=(4.2, dow.max()*0.85),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3d: Monthly heatmap (year × month) ----
recent = cal_df[(cal_df['year'] >= 2010) & (cal_df['year'] <= 2024)]
heatmap_pivot = recent.groupby(['year', 'month']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(heatmap_pivot.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(12))
ax.set_xticklabels(month_names)
ax.set_yticks(range(len(heatmap_pivot.index)))
ax.set_yticklabels(heatmap_pivot.index)
plt.colorbar(im, ax=ax, label='Papers')
ax.set_title('Submission Heatmap: Year × Month (2010–2024)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3e: Top 5 fields — monthly seasonality comparison ----
top5_cats = df['primary_cat'].value_counts().head(5).index.tolist()
print('Top 5 primary categories:', top5_cats)

fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
for i, cat in enumerate(top5_cats):
    sub = cal_df[cal_df['primary_cat'] == cat].groupby('month').size()
    sub = sub.reindex(range(1, 13), fill_value=0)
    axes[i].bar(range(1, 13), sub.values, color=cm.tab10(i), tick_label=month_names)
    axes[i].set_title(cat, fontweight='bold', fontsize=9)
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)
    axes[i].set_ylabel('Papers' if i == 0 else '')

fig.suptitle('Monthly Seasonality by Top 5 Categories', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Takeaways ----
peak_month = monthly.idxmax()
slow_month = monthly.idxmin()
peak_dow   = dow.idxmax()
print('=== Section 3 Takeaways ===')
print(f'Peak submission month:  {month_names[peak_month - 1]} ({monthly.max():,} papers)')
print(f'Slowest submission month: {month_names[slow_month - 1]} ({monthly.min():,} papers)')
print(f'Peak day of week: {dow_names[peak_dow]} ({dow.max():,} papers)')
print(f'Weekend share: {(dow.iloc[5] + dow.iloc[6]) / dow.sum() * 100:.1f}%')

---
## Section 4 — Author Productivity & Cross-Field Reach

**Questions:**
1. Who are the most prolific authors on arXiv?
2. Do highly prolific authors tend to publish across many different fields, or are they deep specialists?
3. What does the distribution of per-author paper counts look like (power law?)  

**How this is unique:** Neither King nor Geon analysed the author dimension. This leverages the `authors_parsed` column — a nested list of `[last, first, suffix]` tuples.

In [ ]:
# Flatten authors — one row per (paper_id, author)
def parse_authors(row):
    paper_id   = row['id']
    primary    = row['primary_cat']
    parsed     = row.get('authors_parsed', [])
    if not isinstance(parsed, list):
        return []
    out = []
    for a in parsed:
        if isinstance(a, list) and len(a) >= 2:
            last  = str(a[0]).strip()
            first = str(a[1]).strip()[:1]   # first initial only
            name  = f'{last}, {first}.' if first else last
            if len(name) > 3:
                out.append({'paper_id': paper_id, 'author': name, 'primary_cat': primary})
    return out

rows_list = []
for _, row in df.dropna(subset=['authors_parsed']).iterrows():
    rows_list.extend(parse_authors(row))

author_df = pd.DataFrame(rows_list)
print(f'Author–paper pairs: {len(author_df):,}')
print(f'Unique authors: {author_df["author"].nunique():,}')

In [ ]:
# Per-author stats
author_stats = author_df.groupby('author').agg(
    paper_count=('paper_id', 'nunique'),
    field_count=('primary_cat', 'nunique'),
    fields=('primary_cat', lambda x: ', '.join(sorted(set(x))))
).sort_values('paper_count', ascending=False)

print('Top 25 most prolific authors:')
print(author_stats.head(25)[['paper_count', 'field_count', 'fields']].to_string())

In [ ]:
# ---- Plot 4a: Log-log distribution of paper counts (power-law check) ----
counts = author_stats['paper_count']
count_dist = counts.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(count_dist.index, count_dist.values, 'o', markersize=3, alpha=0.6, color='steelblue')
ax.set_xlabel('Papers per Author (log scale)')
ax.set_ylabel('Number of Authors (log scale)')
ax.set_title('Author Productivity Distribution — Log-Log Scale\n(Straight line = power law)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4b: Top 30 most prolific authors ----
top30 = author_stats.head(30)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top30.index[::-1], top30['paper_count'][::-1], color='mediumseagreen')
# Annotate field count
for i, (name, row) in enumerate(top30.iloc[::-1].iterrows()):
    ax.text(row['paper_count'] + 1, i, f"{row['field_count']} field{'s' if row['field_count']>1 else ''}",
            va='center', fontsize=8, color='#333333')
ax.set_xlabel('Number of Papers')
ax.set_title('Top 30 Most Prolific Authors on arXiv', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4c: Prolificacy vs. Field breadth scatter ----
productive = author_stats[author_stats['paper_count'] >= 5].copy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(productive['paper_count'], productive['field_count'],
           alpha=0.15, s=8, color='darkorange')

# Highlight top 20 by paper count
top20_a = productive.head(20)
ax.scatter(top20_a['paper_count'], top20_a['field_count'],
           color='red', s=40, zorder=5, label='Top 20 authors')
for name, row in top20_a.head(10).iterrows():
    ax.annotate(name[:20], (row['paper_count'], row['field_count']),
                fontsize=6.5, ha='left', va='bottom',
                xytext=(3, 2), textcoords='offset points')

ax.set_xlabel('Total Papers')
ax.set_ylabel('Number of Distinct Primary Categories')
ax.set_title('Author Productivity vs. Cross-Field Breadth\n(Authors with ≥5 papers)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4d: Field specialisation histogram ----
field_counts = author_stats[author_stats['paper_count'] >= 3]['field_count']
max_f = int(field_counts.quantile(0.99))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(field_counts.clip(upper=max_f), bins=range(1, max_f + 2),
        align='left', color='orchid', edgecolor='white', rwidth=0.8)
ax.set_xlabel('Number of Distinct Primary Categories per Author')
ax.set_ylabel('Number of Authors')
ax.set_title('How Specialised Are arXiv Authors?\n(Authors with ≥3 papers)', fontweight='bold')
pct_specialist = (field_counts == 1).mean() * 100
ax.annotate(f'{pct_specialist:.0f}% publish\nin only 1 field',
            xy=(1, (field_counts == 1).sum()), xytext=(3, (field_counts == 1).sum() * 0.8),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4e: Cross-field authors — top fields they bridge ----
cross_field = author_df[author_df['author'].isin(
    author_stats[author_stats['field_count'] >= 3].index
)]
field_pairs_counter = Counter()
for author, grp in cross_field.groupby('author'):
    fields = sorted(grp['primary_cat'].unique())
    for pair in combinations(fields, 2):
        field_pairs_counter[pair] += 1

top_bridges = field_pairs_counter.most_common(20)
labels_b = [f'{a}  ↔  {b}' for (a, b), _ in top_bridges]
vals_b   = [c for _, c in top_bridges]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels_b[::-1], vals_b[::-1], color='dodgerblue')
ax.set_xlabel('Number of Shared Authors')
ax.set_title('Top 20 Field Pairs Bridged by the Same Authors', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Takeaways ----
print('=== Section 4 Takeaways ===')
print(f'Total unique authors in sample: {author_stats.shape[0]:,}')
print(f'Most prolific author: {author_stats.index[0]} ({author_stats.iloc[0]["paper_count"]} papers, {author_stats.iloc[0]["field_count"]} fields)')
pct_one_field = (author_stats['field_count'] == 1).mean() * 100
print(f'Authors publishing in only 1 primary category: {pct_one_field:.1f}%')
median_papers = author_stats['paper_count'].median()
print(f'Median papers per author: {median_papers:.0f}')

---
## Summary of Findings

| Section | Key Finding |
|---------|-------------|
| **1 — Category Co-occurrence** | A small number of categories (cs.LG, math.MP, stat.ML) act as bridges connecting large portions of the arXiv network. The physics subdomain has the densest internal co-occurrence graph. |
| **2 — LDA Topic Modeling** | 15 latent topics emerge cleanly from abstracts. Most papers have a single dominant topic (high max probability), but a meaningful minority are genuine hybrids. LDA topics align broadly with arXiv category labels but cut across them in interesting ways. |
| **3 — Submission Calendar** | Mondays see the largest submission spikes (authors batching weekend work). January and October are peak months. August and December are notably quieter — matching academic vacation calendars. |
| **4 — Author Productivity** | Author paper-count follows a heavy-tailed (near power-law) distribution. The most prolific authors are overwhelmingly specialists — high paper counts do not strongly predict cross-field breadth. Physics sub-fields produce the highest raw paper counts per author. |